# 🎯 CareerBuddy AI Agent — Run Notebook
**Run cells in order: Cell 1 → Cell 2 → Cell 3 → Cell 4 → Cell 5**

| Cell | Purpose |
|---|---|
| Cell 1 | ✅ Path Setup — always run first |
| Cell 2 | 🚀 Launch Main App (Login UI) |
| Cell 3 | 🧪 Edge Case Tests (Scheduling) |
| Cell 4 | 📊 Full Evaluation (Paper Metrics) |
| Cell 5 | 📋 Career Database Stats |

In [1]:
# ============================================================
# CELL 1 — PATH SETUP
# Always run this first before anything else
# ============================================================

import sys
from pathlib import Path

# Set base directory to wherever this notebook lives
BASE_DIR = Path.cwd()
sys.path.insert(0, str(BASE_DIR))

print('✅ Path set to:', BASE_DIR)
print('📁 Files found:', [f.name for f in BASE_DIR.iterdir() if f.is_dir()])

✅ Path set to: C:\Users\LIBINI\Documents\COLLEGE\2nd YEAR\3rd SEM\AI\ENDSEM_PROJECT\careerbuddy
📁 Files found: ['.ipynb_checkpoints', 'agents', 'assets', 'config', 'models', 'ui', 'utils']


In [2]:
# ============================================================
# CELL 2 — LAUNCH MAIN APPLICATION
# Opens the CareerBuddy login / signup UI
# ============================================================

from models.career_model import CareerBuddyModel
from ui.auth import show_login_page
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import json
import os
from datetime import datetime

USERS_FILE = 'careerbuddy_users.json'

def load_users_db():
    """Load users from JSON file"""
    if os.path.exists(USERS_FILE):
        with open(USERS_FILE, 'r') as f:
            data = json.load(f)
            for username in data:
                data[username]['created'] = datetime.fromisoformat(
                    data[username]['created']
                )
            return data
    return {}

def start_application():
    """Start the CareerBuddy application"""
    global users_db
    users_db = load_users_db()

    display(HTML("""
    <div style='background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
                padding: 40px; border-radius: 15px; text-align: center;
                color: white; margin: 20px 0;
                box-shadow: 0 10px 40px rgba(0,0,0,0.2);'>
        <h1 style='font-size: 48px; margin: 0 0 10px 0;'>🎯 CareerBuddy AI Agent</h1>
        <h3 style='margin: 10px 0; font-weight: 300;'>Mesa Framework — Organised Structure</h3>
        <p style='margin: 20px 0 0 0; font-size: 16px; opacity: 0.9;'>
            Hybrid Agent System with Persistent Storage
        </p>
    </div>
    """))

    with output_area:
        clear_output(wait=True)
        show_login_page(model, users_db, output_area)
    display(output_area)

# Global state — reset every time this cell runs
model       = CareerBuddyModel()
users_db    = {}
output_area = widgets.Output()

start_application()

Output()

In [3]:
# ============================================================
# CELL 3 — EDGE CASE TESTS (Goal-Based Scheduling)
# Tests 8 scheduling scenarios and prints PASS / FAIL
# ============================================================

from agents.modules.goal_based import GoalBasedModule
from config.career_database import CAREER_DATABASE


def test_edge_case_1_high_availability():
    """Test 1: High availability (8 hrs/day) — fewer days expected"""
    print('\n' + '='*70)
    print('TEST 1: High Time Availability (8 hours/day)')
    print('='*70)
    careers = [
        {'career': {'domain': 'Content Writer',
                    'modules': [{'name': 'SEO fundamentals', 'duration': 60},
                                {'name': 'Write articles',   'duration': 90},
                                {'name': 'Create portfolio', 'duration': 120}]}},
        {'career': {'domain': 'Graphic Designer',
                    'modules': [{'name': 'Design principles', 'duration': 90},
                                {'name': 'Learn tools',       'duration': 120},
                                {'name': 'Create portfolio',  'duration': 150}]}},
    ]
    gb       = GoalBasedModule(daily_hours=8.0)
    schedule = gb.generate_daily_schedule(careers)
    total    = sum(len(c['career']['modules']) for c in careers)
    done     = sum(len(d['tasks']) for d in schedule)
    assert done == total, f'Missing tasks! Expected {total}, got {done}'
    print(f'✓ Schedule: {len(schedule)} days, {done}/{total} tasks')
    gb.print_schedule()
    return True


def test_edge_case_2_low_availability():
    """Test 2: Low availability (1 hr/day) — more days, no task dropped"""
    print('\n' + '='*70)
    print('TEST 2: Low Time Availability (1 hour/day)')
    print('='*70)
    careers = [
        {'career': {'domain': 'Content Writer',
                    'modules': [{'name': 'SEO fundamentals', 'duration': 60},
                                {'name': 'Write articles',   'duration': 90},
                                {'name': 'Create portfolio', 'duration': 120}]}},
    ]
    gb       = GoalBasedModule(daily_hours=1.0)
    schedule = gb.generate_daily_schedule(careers)
    total    = sum(len(c['career']['modules']) for c in careers)
    done     = sum(len(d['tasks']) for d in schedule)
    assert done == total, f'Missing tasks! Expected {total}, got {done}'
    print(f'✓ Schedule: {len(schedule)} days, {done}/{total} tasks')
    gb.print_schedule()
    return True


def test_edge_case_3_single_career():
    """Test 3: Single career — all modules scheduled"""
    print('\n' + '='*70)
    print('TEST 3: Single Career Selection')
    print('='*70)
    careers = [
        {'career': {'domain': 'Graphic Designer',
                    'modules': [{'name': 'Design principles', 'duration': 90},
                                {'name': 'Color theory',      'duration': 60},
                                {'name': 'Typography',        'duration': 45},
                                {'name': 'Learn tools',       'duration': 120},
                                {'name': 'Create portfolio',  'duration': 150}]}},
    ]
    gb       = GoalBasedModule(daily_hours=3.0)
    schedule = gb.generate_daily_schedule(careers)
    total    = len(careers[0]['career']['modules'])
    done     = sum(len(d['tasks']) for d in schedule)
    assert done == total, f'Missing tasks! Expected {total}, got {done}'
    print(f'✓ Schedule: {len(schedule)} days, {done}/{total} tasks')
    gb.print_schedule()
    return True


def test_edge_case_4_many_careers():
    """Test 4: Many careers — interleaved correctly"""
    print('\n' + '='*70)
    print('TEST 4: Multiple Career Selection (5 careers)')
    print('='*70)
    careers = [{'career': CAREER_DATABASE[k]}
               for k in ['teaching', 'cooking', 'writing', 'design', 'fitness']
               if k in CAREER_DATABASE]
    gb       = GoalBasedModule(daily_hours=4.0)
    schedule = gb.generate_daily_schedule(careers)
    total    = sum(len(c['career']['modules']) for c in careers)
    done     = sum(len(d['tasks']) for d in schedule)
    assert done == total, f'Missing tasks! Expected {total}, got {done}'
    day1_careers = set(t['career'] for t in schedule[0]['tasks'])
    print(f'✓ Schedule: {len(schedule)} days, {done}/{total} tasks')
    print(f'✓ Day 1 variety: {len(day1_careers)} different careers')
    gb.print_schedule()
    return True


def test_edge_case_5_empty_input():
    """Test 5: Empty input — handled gracefully"""
    print('\n' + '='*70)
    print('TEST 5: Empty / Invalid Input Handling')
    print('='*70)
    gb = GoalBasedModule(daily_hours=2.0)
    s1 = gb.generate_daily_schedule([])
    assert s1 == [], 'Empty list should return []'
    print('✓ Empty career list handled correctly')
    s2 = gb.generate_daily_schedule([{'career': {'domain': 'Test', 'modules': []}}])
    assert len(s2) == 0, 'Career with no modules should return []'
    print('✓ Career with no modules handled correctly')
    return True


def test_edge_case_6_large_task():
    """Test 6: Task bigger than daily limit — placed on its own day"""
    print('\n' + '='*70)
    print('TEST 6: Task Larger Than Daily Capacity')
    print('='*70)
    careers = [
        {'career': {'domain': 'Content Creator',
                    'modules': [{'name': 'Small task',  'duration': 30},
                                {'name': 'HUGE task',   'duration': 300},
                                {'name': 'Normal task', 'duration': 60}]}},
    ]
    gb       = GoalBasedModule(daily_hours=2.0)
    schedule = gb.generate_daily_schedule(careers)
    done     = sum(len(d['tasks']) for d in schedule)
    assert done == 3, f'Expected 3, got {done}'
    large_day = next((d for d in schedule
                      for t in d['tasks'] if t['duration'] == 300), None)
    assert large_day is not None, 'Large task not found'
    print(f'✓ All 3 tasks scheduled — large task on Day {large_day["day"]}')
    gb.print_schedule()
    return True


def test_edge_case_7_uneven_modules():
    """Test 7: Uneven module counts — workload balanced"""
    print('\n' + '='*70)
    print('TEST 7: Uneven Module Distribution')
    print('='*70)
    careers = [
        {'career': {'domain': 'Simple Career',
                    'modules': [{'name': 'Task 1', 'duration': 60},
                                {'name': 'Task 2', 'duration': 60}]}},
        {'career': {'domain': 'Complex Career',
                    'modules': [{'name': f'Task {i}', 'duration': 45}
                                for i in range(1, 11)]}},
    ]
    gb       = GoalBasedModule(daily_hours=3.0)
    schedule = gb.generate_daily_schedule(careers)
    done     = sum(len(d['tasks']) for d in schedule)
    assert done == 12, f'Expected 12, got {done}'
    print(f'✓ All 12 tasks scheduled across {len(schedule)} days')
    gb.print_schedule()
    return True


def test_progress_tracking():
    """Test 8: Progress tracking — 0% → partial → 100%"""
    print('\n' + '='*70)
    print('TEST 8: Progress Tracking')
    print('='*70)
    careers = [
        {'career': {'domain': 'Test Career',
                    'modules': [{'name': 'Task 1', 'duration': 60},
                                {'name': 'Task 2', 'duration': 60},
                                {'name': 'Task 3', 'duration': 60}]}},
    ]
    gb       = GoalBasedModule(daily_hours=2.0)
    schedule = gb.generate_daily_schedule(careers)
    p0       = gb.track_progress()
    assert p0['completed'] == 0 and p0['percentage'] == 0
    print(f'✓ Initial: {p0["completed"]}/{p0["total"]} (0%)')

    first_id = schedule[0]['tasks'][0]['id']
    gb.update_progress(first_id, True)
    p1 = gb.track_progress()
    assert p1['completed'] == 1
    print(f'✓ After 1 task: {p1["completed"]}/{p1["total"]} ({p1["percentage"]:.1f}%)')

    for day in schedule:
        for task in day['tasks']:
            gb.update_progress(task['id'], True)
    pf = gb.track_progress()
    assert pf['goal_achieved'] == True
    print(f'✓ Complete: {pf["completed"]}/{pf["total"]} (100%) 🎉')
    return True


# ── Run all tests ────────────────────────────────────────────
def run_all_tests():
    print('\n' + '='*70)
    print('GOAL-BASED SCHEDULING — COMPREHENSIVE EDGE CASE TESTS')
    print('='*70)
    tests = [
        ('High Time Availability',  test_edge_case_1_high_availability),
        ('Low Time Availability',   test_edge_case_2_low_availability),
        ('Single Career',           test_edge_case_3_single_career),
        ('Multiple Careers',        test_edge_case_4_many_careers),
        ('Empty Input',             test_edge_case_5_empty_input),
        ('Large Task',              test_edge_case_6_large_task),
        ('Uneven Modules',          test_edge_case_7_uneven_modules),
        ('Progress Tracking',       test_progress_tracking),
    ]
    passed = failed = 0
    for name, fn in tests:
        try:
            fn()
            passed += 1
            print(f'✅ {name}: PASSED')
        except Exception as e:
            failed += 1
            print(f'❌ {name}: FAILED — {e}')
    print('\n' + '='*70)
    print(f'RESULTS: {passed} passed, {failed} failed')
    print('='*70)

run_all_tests()


GOAL-BASED SCHEDULING — COMPREHENSIVE EDGE CASE TESTS

TEST 1: High Time Availability (8 hours/day)
✓ Schedule: 2 days, 6/6 tasks

  Learning Schedule Summary
  Total Days   : 2
  Total Tasks  : 6
  Total Time   : 630 min (10.5 hrs)
  Optimality   : 25.90%
  Utilization  : 65.60%

Day 1  (480 mins / 8.0 hrs):
  ⬜ [Content Writer] SEO fundamentals (60 min)
  ⬜ [Graphic Designer] Design principles (90 min)
  ⬜ [Graphic Designer] Learn tools (120 min)
  ⬜ [Content Writer] Write articles (90 min)
  ⬜ [Content Writer] Create portfolio (120 min)
-------------------------------------------------------
Day 2  (150 mins / 2.5 hrs):
  ⬜ [Graphic Designer] Create portfolio (150 min)
-------------------------------------------------------
✅ High Time Availability: PASSED

TEST 2: Low Time Availability (1 hour/day)
✓ Schedule: 3 days, 3/3 tasks

  Learning Schedule Summary
  Total Days   : 3
  Total Tasks  : 3
  Total Time   : 270 min (4.5 hrs)
  Optimality   : 66.70%
  Utilization  : 150.00%

Day

In [4]:
# ============================================================
# CELL 4 — FULL QUANTITATIVE EVALUATION (Paper Metrics)
# Runs: Precision/Recall/F1, Scheduling Comparison,
#       User Satisfaction (N=50), Fairness, Scalability
# ============================================================

from evaluation import run_full_evaluation

results = run_full_evaluation(daily_hours=2.0)

# Quick summary print
print('\n📌 QUICK SUMMARY FOR PAPER')
print('─' * 40)
rec = results['recommendation']
sat = results['satisfaction']
sb  = results['scalability']
fair = results['fairness']
print(f"  Avg Precision    : {rec['avg_precision']:.2%}")
print(f"  Avg Recall       : {rec['avg_recall']:.2%}")
print(f"  Avg F1-Score     : {rec['avg_f1']:.2%}")
print(f"  Mean Satisfaction: {sat['mean']} / 10  (N={sat['n']})")
print(f"  Fairness Spread  : {max(v['avg'] for v in fair.values()) - min(v['avg'] for v in fair.values()):.2f}")
print(f"  Max Schedule Time: {sb[-1]['ms']} ms  ({sb[-1]['n_careers']} careers)")


████████████████████████████████████████████████████████████████████████
   CareerBuddy v2 — Full Quantitative Evaluation Suite
   2026-04-12 22:27:52
████████████████████████████████████████████████████████████████████████

  RECOMMENDATION EVALUATION — Precision / Recall / F1
  Profiles evaluated  :  10
  Avg Precision       :  100.00%
  Avg Recall          :  70.00%
  Avg F1-Score        :  80.00%
--------------------------------------------------------------
  [graduate_teaching_remote]
    Expected  : {'Online Home Tutor', 'Content Writer'}
    Predicted : {'Online Home Tutor'}
    P=1.00  R=0.50  F1=0.67
  [graduate_design_flexible]
    Expected  : {'Social Media Manager', 'Graphic Designer'}
    Predicted : {'Graphic Designer'}
    P=1.00  R=0.50  F1=0.67
  [graduate_writing_full-time]
    Expected  : {'Freelance Translator / Transcriptionist', 'Content Writer'}
    Predicted : {'Content Writer'}
    P=1.00  R=0.50  F1=0.67
  [high-school_cooking_flexible]
    Expected  : {'Foo

In [5]:
# ============================================================
# CELL 5 — CAREER DATABASE STATS
# Shows overview of the expanded 15-career database
# ============================================================

from config.career_database import get_career_statistics, CAREER_DATABASE

stats = get_career_statistics()

print('📋 Career Database Overview')
print('=' * 45)
print(f"  Total Careers     : {stats['total_careers']}")
print(f"  Total Modules     : {stats['total_modules']}")
print(f"  Avg Modules/Career: {stats['avg_modules_per_career']}")
print(f"  Total Duration    : {stats['total_duration_minutes']} min"
      f" ({stats['total_duration_minutes']/60:.1f} hrs)")
print(f"  Avg Duration/Career: {stats['avg_duration_per_career_minutes']} min")
print(f"  Difficulty Split  : {stats['difficulty_distribution']}")
print()
print('  All careers:')
for i, (key, val) in enumerate(CAREER_DATABASE.items(), 1):
    diff  = val.get('difficulty_level', 'n/a')
    income = val.get('avg_monthly_income_inr', 0)
    print(f"  {i:>2}. {val['domain']:<40} [{diff}]  ₹{income:,}/mo")

📋 Career Database Overview
  Total Careers     : 15
  Total Modules     : 75
  Avg Modules/Career: 5.0
  Total Duration    : 4995 min (83.2 hrs)
  Avg Duration/Career: 333.0 min
  Difficulty Split  : {'beginner': 9, 'intermediate': 6}

  All careers:
   1. Online Home Tutor                        [beginner]  ₹15,000/mo
   2. Food Content Creator                     [beginner]  ₹12,000/mo
   3. Content Writer                           [beginner]  ₹18,000/mo
   4. Graphic Designer                         [intermediate]  ₹20,000/mo
   5. Handmade Products Seller                 [beginner]  ₹10,000/mo
   6. Online Fitness Coach                     [intermediate]  ₹22,000/mo
   7. Freelance Data Entry Specialist          [beginner]  ₹8,000/mo
   8. Freelance Translator / Transcriptionist  [intermediate]  ₹16,000/mo
   9. Social Media Manager                     [beginner]  ₹14,000/mo
  10. Home-Based Bookkeeper                    [intermediate]  ₹18,000/mo
  11. E-Commerce Reseller         